In [ ]:
import os
import pandas as pd

# make a folder on Colab's temporary disk
os.makedirs("/content/ku", exist_ok=True)

# fetch the two small spreadsheets straight from Zenodo
!wget -q -O /content/ku/metadata.csv "https://zenodo.org/records/17333317/files/metadata.csv?download=1"
!wget -q -O /content/ku/metadata_with_patient_level_splits.csv "https://zenodo.org/records/17333317/files/metadata_with_patient_level_splits.csv?download=1"
!ls -la /content/ku

# read each one and show its column names and first 5 rows
for f in ["metadata.csv", "metadata_with_patient_level_splits.csv"]:
    d = pd.read_csv("/content/ku/" + f)
    print("\n==", f, d.shape)
    print(d.columns.tolist())
    print(d.head(5).to_string(index=False))

total 3912
drwxr-xr-x 2 root root    4096 Nov 19 00:00 .
drwxr-xr-x 1 root root    4096 Nov 19 00:00 ..
-rw-r--r-- 1 root root 1652736 Nov 19 00:00 metadata.csv
-rw-r--r-- 1 root root 2342655 Nov 19 00:00 metadata_with_patient_level_splits.csv

== metadata.csv (31484, 3)
['patient_id', 'image_name', 'path']
patient_id         image_name                  path
1604138765 1604138765_102.jpg train/band_neutrophil
1604138765 1604138765_107.jpg train/band_neutrophil
1604144357 1604144357_017.jpg train/band_neutrophil
1604144357 1604144357_050.jpg train/band_neutrophil
1604144357 1604144357_060.jpg train/band_neutrophil

== metadata_with_patient_level_splits.csv (31484, 5)
['patient_id', 'image_name', 'path', 'cell_type', 'split']
patient_id         image_name                  path       cell_type      split
1604138765 1604138765_102.jpg train/band_neutrophil band_neutrophil validation
1604138765 1604138765_107.jpg train/band_neutrophil band_neutrophil validation
1604144357 1604144357_017.jpg

In [ ]:
d = pd.read_csv("/content/ku/metadata_with_patient_level_splits.csv")
print("total patients:", d["patient_id"].nunique())
print("patients appearing in more than one split:", int((d.groupby("patient_id")["split"].nunique() > 1).sum()))

cnt = d.groupby("cell_type").agg(images=("image_name", "size"),
                                  patients=("patient_id", "nunique")).sort_values("images", ascending=False)
print("\n", cnt.to_string())
print("\n", pd.crosstab(d["cell_type"], d["split"]).to_string())

total patients: 276
patients appearing in more than one split: 0

                       images  patients
cell_type                             
segmented_neutrophil   16256       243
lymphocyte              5762       241
giant_platelet          2835       217
eosinophil              1862       175
monocyte                1381       240
reactive_lymphocyte     1038        48
blast                    814        12
basophil                 546       121
metamyelocyte            262        25
erythroblast             243        12
band_neutrophil          199        31
platelet_cluster         188        16
myelocyte                 98        33

 split                 test  train  validation
cell_type                                    
band_neutrophil         32    151          16
basophil                46    479          21
blast                   86    728           0
eosinophil             140   1663          59
erythroblast            16    226           1
giant_platelet         6

In [ ]:
import os, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
base = "/content/drive/MyDrive/WBC-Folder/"

d = pd.read_csv("/content/ku/metadata_with_patient_level_splits.csv")
cmap = {"segmented_neutrophil": "neutrophil", "lymphocyte": "lymphocyte", "monocyte": "monocyte",
        "eosinophil": "eosinophil", "basophil": "basophil", "erythroblast": "erythroblast",
        "metamyelocyte": "ig", "myelocyte": "ig"}
d["label"] = d["cell_type"].map(cmap)
m = d[d["label"].notna()].copy()
classes7 = sorted(set(cmap.values()))
patients = np.array(sorted(m["patient_id"].unique()))
print(len(patients), "patients have at least one of the 7 classes")

chosen = None
for seed in range(500):
    perm = np.random.RandomState(seed).permutation(patients)
    n_adapt = int(round(0.3 * len(perm)))
    adapt_p, test_p = set(perm[:n_adapt]), set(perm[n_adapt:])
    ok = all(
        m[m.label == c].patient_id.isin(adapt_p).sum() >= 30 and
        m[m.label == c].patient_id.isin(test_p).sum() >= 100
        for c in classes7)
    if ok:
        chosen = seed
        break
print("patient split seed:", chosen, "| adaptation patients:", len(adapt_p), "| test patients:", len(test_p))

test = pd.concat([m[(m.label == c) & m.patient_id.isin(test_p)].sample(100, random_state=42) for c in classes7])
adapt = pd.concat([m[(m.label == c) & m.patient_id.isin(adapt_p)].sample(14, random_state=7) for c in classes7])
print(len(test), len(adapt))
print("patient overlap:", len(set(test.patient_id) & set(adapt.patient_id)))
print("image overlap:", len(set(test.image_name) & set(adapt.image_name)))
print(test.label.value_counts().to_dict())
print(adapt.label.value_counts().to_dict())
print("distinct patients behind each test class:", test.groupby("label").patient_id.nunique().to_dict())
print("distinct patients behind each adaptation class:", adapt.groupby("label").patient_id.nunique().to_dict())

for obj, name in [(test, "ku_frozen_test_700.csv"), (adapt, "ku_adapt_98.csv")]:
    if os.path.exists(base + name):
        print(name, "already exists, not overwriting")
    else:
        obj.to_csv(base + name, index=False)
        print("saved", name)

Mounted at /content/drive
274 patients have at least one of the 7 classes
patient split seed: 7 | adaptation patients: 82 | test patients: 192
700 98
patient overlap: 0
image overlap: 0
{'basophil': 100, 'eosinophil': 100, 'erythroblast': 100, 'ig': 100, 'lymphocyte': 100, 'monocyte': 100, 'neutrophil': 100}
{'basophil': 14, 'eosinophil': 14, 'erythroblast': 14, 'ig': 14, 'lymphocyte': 14, 'monocyte': 14, 'neutrophil': 14}
distinct patients behind each test class: {'basophil': 28, 'eosinophil': 19, 'erythroblast': 8, 'ig': 22, 'lymphocyte': 73, 'monocyte': 69, 'neutrophil': 80}
distinct patients behind each adaptation class: {'basophil': 14, 'eosinophil': 12, 'erythroblast': 2, 'ig': 11, 'lymphocyte': 12, 'monocyte': 12, 'neutrophil': 13}
saved ku_frozen_test_700.csv
saved ku_adapt_98.csv


In [ ]:
import os, shutil, zipfile
import pandas as pd

base = "/content/drive/MyDrive/WBC-Folder/"
test = pd.read_csv(base + "ku_frozen_test_700.csv"); test["split2"] = "test"
adapt = pd.read_csv(base + "ku_adapt_98.csv"); adapt["split2"] = "adapt"
want = pd.concat([test, adapt]).reset_index(drop=True)

zp = "/content/ku/dataset.zip"
if not os.path.exists(zp):
    os.system('wget -q -O /content/ku/dataset.zip "https://zenodo.org/records/17333317/files/dataset.zip?download=1"')
print("zip size:", round(os.path.getsize(zp) / 1e6, 1), "MB")

dest_root = "/content/drive/MyDrive/ku/images_used/"
want["local"] = dest_root + want["label"] + "/" + want["image_name"]
missing = 0
with zipfile.ZipFile(zp) as z:
    names = [n for n in z.namelist() if n.lower().endswith(".jpg") and "__MACOSX" not in n]
    print(len(names), "images in zip | example:", names[:2])
    bymem = {}
    for n in names:
        bymem.setdefault(os.path.basename(n), []).append(n)
    print("duplicate file names in zip:", sum(len(v) > 1 for v in bymem.values()))
    for _, r in want.iterrows():
        cand = bymem.get(r["image_name"])
        if not cand:
            missing += 1
            continue
        os.makedirs(os.path.dirname(r["local"]), exist_ok=True)
        if not (os.path.exists(r["local"]) and os.path.getsize(r["local"]) > 0):
            with z.open(cand[0]) as src, open(r["local"], "wb") as out:
                shutil.copyfileobj(src, out)
have = sum(os.path.exists(p) and os.path.getsize(p) > 0 for p in want["local"])
print("images saved:", have, "of", len(want), "| missing in zip:", missing)
print(want.groupby(["split2", "label"]).size().to_string())

out = base + "ku_want_798.csv"
if os.path.exists(out):
    print("list already exists, not overwriting")
else:
    want.to_csv(out, index=False)
    print("saved", out)

zip size: 564.7 MB
31488 images in zip | example: ['dataset/test/Band Neutrophil/1604144357_052.jpg', 'dataset/test/Band Neutrophil/1604144357_054.jpg']
duplicate file names in zip: 0
images saved: 798 of 798 | missing in zip: 0
split2  label       
adapt   basophil         14
        eosinophil       14
        erythroblast     14
        ig               14
        lymphocyte       14
        monocyte         14
        neutrophil       14
test    basophil        100
        eosinophil      100
        erythroblast    100
        ig              100
        lymphocyte      100
        monocyte        100
        neutrophil      100
saved /content/drive/MyDrive/WBC-Folder/ku_want_798.csv


In [ ]:
import io, zipfile, os
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/WBC-Folder/"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
classes = ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

norm = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
plain_tfm = transforms.Compose([transforms.Resize((224, 224)), norm])

def shrink_pad(img, s, size=224):
    inner = int(size * s)
    small = img.resize((inner, inner))
    arr = np.array(img)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    bg = tuple(int(v) for v in np.median(border, axis=0))
    canvas = Image.new("RGB", (size, size), bg)
    canvas.paste(small, ((size - inner) // 2, (size - inner) // 2))
    return canvas

def bg_color(arr):
    flat = arr.reshape(-1, 3).astype("float32")
    lum = flat.mean(axis=1)
    return np.median(flat[lum >= np.percentile(lum, 90)], axis=0)

def bg_match(img, ref_bg):
    arr = np.array(img).astype("float32")
    gain = ref_bg / np.maximum(bg_color(arr), 1)
    return Image.fromarray(np.clip(arr * gain, 0, 255).astype("uint8"))

with zipfile.ZipFile(base + "PBC_dataset_normal_DIB.zip") as z:
    hit = [n for n in z.namelist() if "BNE_56697" in n and "__MACOSX" not in n][0]
    refA = bg_color(np.array(Image.open(io.BytesIO(z.read(hit))).convert("RGB")))
print("background A:", refA.round(0))

def prep_plain(p): return plain_tfm(Image.open(p).convert("RGB"))
def prep_fix(p): return norm(shrink_pad(bg_match(Image.open(p).convert("RGB"), refA), 0.65))

want = pd.read_csv(base + "ku_want_798.csv")
adapt = want[want["split2"] == "adapt"].reset_index(drop=True)
y = torch.tensor([classes.index(l) for l in adapt["label"]])
Xs = {"plain": torch.stack([prep_plain(p) for p in adapt["local"]]),
      "fixes": torch.stack([prep_fix(p) for p in adapt["local"]])}
print(Xs["plain"].shape, adapt["label"].value_counts().to_dict())

rng = np.random.RandomState(0)
fold = np.zeros(len(adapt), dtype=int)
for lab in adapt["label"].unique():
    idx = np.where(adapt["label"].values == lab)[0]
    rng.shuffle(idx)
    fold[idx] = np.arange(len(idx)) % 7

def fresh_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(base + "resnet18_wbc_v2.pt", map_location=device))
    return m.to(device)

def train_eval(X, setting, tr, va, epochs=15, bs=16, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    m = fresh_model()
    for p in m.parameters():
        p.requires_grad = False
    fc_params = list(m.fc.parameters())
    for p in fc_params:
        p.requires_grad = True
    groups = [{"params": fc_params, "lr": 1e-3}]
    if setting == "B":
        l4 = list(m.layer4.parameters())
        for p in l4:
            p.requires_grad = True
        groups.append({"params": l4, "lr": 1e-4})
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    m.eval()  # keeps BatchNorm statistics frozen
    for ep in range(epochs):
        perm = torch.randperm(len(tr))
        for i in range(0, len(tr), bs):
            b = tr[perm[i:i + bs]]
            xb, yb = X[b].to(device), y[b].to(device)
            if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[3])
            if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[2])
            loss = lossf(m(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return m(X[va].to(device)).argmax(1).cpu()

def report(name, pred):
    ok = (pred == y).numpy()
    rec = {classes[c]: round(float(ok[(y == c).numpy()].mean()), 2) for c in y.unique().tolist()}
    print(f"{name}: accuracy {ok.mean():.3f} | recall {rec}")

m0 = fresh_model().eval()
for pre in ["plain", "fixes"]:
    with torch.no_grad():
        p0 = m0(Xs[pre].to(device)).argmax(1).cpu()
    report(f"no adaptation, {pre}", p0)

for pre in ["plain", "fixes"]:
    for setting in ["A", "B"]:
        pred_all = torch.zeros(len(y), dtype=torch.long)
        for k in range(7):
            va = torch.tensor(np.where(fold == k)[0])
            tr = torch.tensor(np.where(fold != k)[0])
            pred_all[va] = train_eval(Xs[pre], setting, tr, va)
        report(f"CV {pre} + setting {setting}", pred_all)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cpu
background A: [255. 228. 203.]
torch.Size([98, 3, 224, 224]) {'basophil': 14, 'eosinophil': 14, 'erythroblast': 14, 'ig': 14, 'lymphocyte': 14, 'monocyte': 14, 'neutrophil': 14}
no adaptation, plain: accuracy 0.755 | recall {'basophil': 0.57, 'eosinophil': 1.0, 'erythroblast': 1.0, 'ig': 0.64, 'lymphocyte': 0.29, 'monocyte': 0.86, 'neutrophil': 0.93}
no adaptation, fixes: accuracy 0.561 | recall {'basophil': 0.29, 'eosinophil': 0.71, 'erythroblast': 1.0, 'ig': 0.86, 'lymphocyte': 0.07, 'monocyte': 0.43, 'neutrophil': 0.57}
CV plain + setting A: accuracy 0.949 | recall {'basophil': 1.0, 'eosinophil': 1.0, 'erythroblast': 1.0, 'ig': 0.86, 'lymphocyte': 0.93, 'monocyte': 0.86, 'neutrophil': 1.0}
CV plain + setting B: accuracy 0.949 | recall {'basophil': 1.0, 'eosinophil': 1.0, 'erythroblast': 1.0, 'ig': 1.0, 'lymphocyte': 0.86, 'monocyte': 0.79, 'neu

In [1]:
import io, zipfile, os
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/WBC-Folder/"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
classes = ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

norm = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
plain_tfm = transforms.Compose([transforms.Resize((224, 224)), norm])

def shrink_pad(img, s, size=224):
    inner = int(size * s)
    small = img.resize((inner, inner))
    arr = np.array(img)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    bg = tuple(int(v) for v in np.median(border, axis=0))
    canvas = Image.new("RGB", (size, size), bg)
    canvas.paste(small, ((size - inner) // 2, (size - inner) // 2))
    return canvas

def bg_color(arr):
    flat = arr.reshape(-1, 3).astype("float32")
    lum = flat.mean(axis=1)
    return np.median(flat[lum >= np.percentile(lum, 90)], axis=0)

def bg_match(img, ref_bg):
    arr = np.array(img).astype("float32")
    gain = ref_bg / np.maximum(bg_color(arr), 1)
    return Image.fromarray(np.clip(arr * gain, 0, 255).astype("uint8"))

with zipfile.ZipFile(base + "PBC_dataset_normal_DIB.zip") as z:
    hit = [n for n in z.namelist() if "BNE_56697" in n and "__MACOSX" not in n][0]
    refA = bg_color(np.array(Image.open(io.BytesIO(z.read(hit))).convert("RGB")))
print("background A:", refA.round(0))

def prep_plain(p): return plain_tfm(Image.open(p).convert("RGB"))
def prep_fix(p): return norm(shrink_pad(bg_match(Image.open(p).convert("RGB"), refA), 0.65))

want = pd.read_csv(base + "ku_want_798.csv")
adapt = want[want["split2"] == "adapt"].reset_index(drop=True)
y = torch.tensor([classes.index(l) for l in adapt["label"]])
Xs = {"plain": torch.stack([prep_plain(p) for p in adapt["local"]]),
      "fixes": torch.stack([prep_fix(p) for p in adapt["local"]])}
print(Xs["plain"].shape, adapt["label"].value_counts().to_dict())

rng = np.random.RandomState(0)
fold = np.zeros(len(adapt), dtype=int)
for lab in adapt["label"].unique():
    idx = np.where(adapt["label"].values == lab)[0]
    rng.shuffle(idx)
    fold[idx] = np.arange(len(idx)) % 7

def fresh_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(base + "resnet18_wbc_v2.pt", map_location=device))
    return m.to(device)

def train_eval(X, setting, tr, va, epochs=15, bs=16, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    m = fresh_model()
    for p in m.parameters():
        p.requires_grad = False
    fc_params = list(m.fc.parameters())
    for p in fc_params:
        p.requires_grad = True
    groups = [{"params": fc_params, "lr": 1e-3}]
    if setting == "B":
        l4 = list(m.layer4.parameters())
        for p in l4:
            p.requires_grad = True
        groups.append({"params": l4, "lr": 1e-4})
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    m.eval()  # keeps BatchNorm statistics frozen
    for ep in range(epochs):
        perm = torch.randperm(len(tr))
        for i in range(0, len(tr), bs):
            b = tr[perm[i:i + bs]]
            xb, yb = X[b].to(device), y[b].to(device)
            if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[3])
            if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[2])
            loss = lossf(m(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return m(X[va].to(device)).argmax(1).cpu()

def report(name, pred):
    ok = (pred == y).numpy()
    rec = {classes[c]: round(float(ok[(y == c).numpy()].mean()), 2) for c in y.unique().tolist()}
    print(f"{name}: accuracy {ok.mean():.3f} | recall {rec}")

m0 = fresh_model().eval()
for pre in ["plain", "fixes"]:
    with torch.no_grad():
        p0 = m0(Xs[pre].to(device)).argmax(1).cpu()
    report(f"no adaptation, {pre}", p0)

for pre in ["plain", "fixes"]:
    for setting in ["A", "B"]:
        pred_all = torch.zeros(len(y), dtype=torch.long)
        for k in range(7):
            va = torch.tensor(np.where(fold == k)[0])
            tr = torch.tensor(np.where(fold != k)[0])
            pred_all[va] = train_eval(Xs[pre], setting, tr, va)
        report(f"CV {pre} + setting {setting}", pred_all)

Mounted at /content/drive
device: cuda
background A: [255. 228. 203.]
torch.Size([98, 3, 224, 224]) {'basophil': 14, 'eosinophil': 14, 'erythroblast': 14, 'ig': 14, 'lymphocyte': 14, 'monocyte': 14, 'neutrophil': 14}
no adaptation, plain: accuracy 0.755 | recall {'basophil': 0.57, 'eosinophil': 1.0, 'erythroblast': 1.0, 'ig': 0.64, 'lymphocyte': 0.29, 'monocyte': 0.86, 'neutrophil': 0.93}
no adaptation, fixes: accuracy 0.561 | recall {'basophil': 0.29, 'eosinophil': 0.71, 'erythroblast': 1.0, 'ig': 0.86, 'lymphocyte': 0.07, 'monocyte': 0.43, 'neutrophil': 0.57}
CV plain + setting A: accuracy 0.949 | recall {'basophil': 1.0, 'eosinophil': 1.0, 'erythroblast': 1.0, 'ig': 0.86, 'lymphocyte': 0.93, 'monocyte': 0.86, 'neutrophil': 1.0}
CV plain + setting B: accuracy 0.949 | recall {'basophil': 1.0, 'eosinophil': 1.0, 'erythroblast': 1.0, 'ig': 1.0, 'lymphocyte': 0.86, 'monocyte': 0.79, 'neutrophil': 1.0}
CV fixes + setting A: accuracy 0.857 | recall {'basophil': 0.79, 'eosinophil': 0.93, 'e

In [2]:
out = base + "ku_baseline_v2_test700.csv"
assert not os.path.exists(out), "file exists, do not overwrite"

test = want[want["split2"] == "test"].reset_index(drop=True)
print(len(test), test["label"].value_counts().to_dict())

yt = torch.tensor([classes.index(l) for l in test["label"]])
Xt = torch.stack([prep_plain(p) for p in test["local"]])

m0 = fresh_model().eval()
with torch.no_grad():
    pt = torch.cat([m0(Xt[i:i+100].to(device)).argmax(1).cpu() for i in range(0, len(Xt), 100)])

ok = (pt == yt).numpy()
rec = {classes[c]: round(float(ok[(yt == c).numpy()].mean()), 2) for c in yt.unique().tolist()}
print(f"plain v2, 0 adaptation images, KU test 700: accuracy {ok.mean():.3f}")
print("recall:", rec)
print("predicted share:", {classes[c]: round(float((pt == c).float().mean()), 3) for c in range(8)})

pd.DataFrame({"image_name": test["image_name"], "label": test["label"],
              "pred": [classes[i] for i in pt.tolist()]}).to_csv(out, index=False)

700 {'basophil': 100, 'eosinophil': 100, 'erythroblast': 100, 'ig': 100, 'lymphocyte': 100, 'monocyte': 100, 'neutrophil': 100}
plain v2, 0 adaptation images, KU test 700: accuracy 0.810
recall: {'basophil': 0.66, 'eosinophil': 0.97, 'erythroblast': 1.0, 'ig': 0.55, 'lymphocyte': 0.59, 'monocyte': 0.99, 'neutrophil': 0.91}
predicted share: {'basophil': 0.094, 'eosinophil': 0.139, 'erythroblast': 0.266, 'ig': 0.083, 'lymphocyte': 0.09, 'monocyte': 0.194, 'neutrophil': 0.134, 'platelet': 0.0}


In [3]:
out_pred = base + "ku_adapted_plainA_test700.csv"
out_w = base + "resnet18_wbc_v2_ku_plainA.pt"
assert not os.path.exists(out_pred) and not os.path.exists(out_w), "file exists, do not overwrite"

torch.manual_seed(0); np.random.seed(0)
m = fresh_model()
for p in m.parameters():
    p.requires_grad = False
fc_params = list(m.fc.parameters())
for p in fc_params:
    p.requires_grad = True
opt = torch.optim.AdamW([{"params": fc_params, "lr": 1e-3}], weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()
X = Xs["plain"]
m.eval()  # BatchNorm frozen
for ep in range(15):
    perm = torch.randperm(len(X))
    for i in range(0, len(X), 16):
        b = perm[i:i + 16]
        xb, yb = X[b].to(device), y[b].to(device)
        if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[3])
        if np.random.rand() < 0.5: xb = torch.flip(xb, dims=[2])
        loss = lossf(m(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    pa = torch.cat([m(Xt[i:i+100].to(device)).argmax(1).cpu() for i in range(0, len(Xt), 100)])

ok = (pa == yt).numpy()
rec = {classes[c]: round(float(ok[(yt == c).numpy()].mean()), 2) for c in yt.unique().tolist()}
print(f"adapted plain+A, 98 adaptation images, KU test 700: accuracy {ok.mean():.3f}")
print("recall:", rec)
print("predicted share:", {classes[c]: round(float((pa == c).float().mean()), 3) for c in range(8)})

pd.DataFrame({"image_name": test["image_name"], "label": test["label"],
              "pred": [classes[i] for i in pa.tolist()]}).to_csv(out_pred, index=False)
torch.save(m.state_dict(), out_w)

adapted plain+A, 98 adaptation images, KU test 700: accuracy 0.929
recall: {'basophil': 0.98, 'eosinophil': 0.99, 'erythroblast': 0.86, 'ig': 0.89, 'lymphocyte': 0.98, 'monocyte': 0.85, 'neutrophil': 0.95}
predicted share: {'basophil': 0.14, 'eosinophil': 0.143, 'erythroblast': 0.124, 'ig': 0.161, 'lymphocyte': 0.161, 'monocyte': 0.131, 'neutrophil': 0.139, 'platelet': 0.0}
